# Federated Dataset Preparation — IID, Optimized V2

This notebook prepares the **same CICIoT2023 data pipeline used by the completed centralized baseline** for federated learning.

It keeps the centralized-baseline choices aligned:

- fixed class order: **34 classes**
- input features: **39**
- seed: **42**
- train/test split: **80/20, stratified**
- scaling: **StandardScaler fitted only on training data**
- default dataset fraction: **1.00**, so the FL experiment uses the same full-data scope as the completed centralized baseline

The only new step is that the training split is divided into **10 IID clients**. The test split stays global and untouched.

## Kaggle input for this notebook

Add only the raw Kaggle dataset:

**`mashrurfaiyaz28/ciot2023-2`**

Then enable a GPU only if you want; this preparation notebook itself is mainly CPU/RAM/disk work.

## Output of this notebook

The folder `/kaggle/working/fl_prepared_iid_v2/` will contain the arrays, split indices, client partitions, scaler statistics, and metadata needed by `fedavg-iid-optimized-v2.ipynb`.

After this notebook finishes, click **Save Version** on Kaggle so its output can be added as an input to the FedAvg notebook.


In [ ]:
# Cell 02: Imports
from pathlib import Path
import gc
import json
import os
import platform
import random
import time

import numpy as np
import pandas as pd
import psutil

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
# Cell 03: Configuration — aligned with centralized-baseline-markp.ipynb
EXPERIMENT_NAME = "fl_dataset_preparation_iid_v2"
ARTIFACT_TYPE = "fl_prepared_iid_v2"

# Use 1.00 for the fair full-data comparison with the completed centralized baseline.
# For a quick smoke test only, you may temporarily use 0.10 or 0.25.
dataset_frac = 1.00

SEED = 42
TEST_SIZE = 0.20
NUM_CLIENTS = 10
SCALER_CHUNK_SIZE = 250_000

if not (0 < dataset_frac <= 1):
    raise ValueError("dataset_frac must satisfy 0 < dataset_frac <= 1")
if NUM_CLIENTS < 2:
    raise ValueError("NUM_CLIENTS must be at least 2")

DATA_DIR_CANDIDATES = [
    Path("/kaggle/input/ciot2023-2"),
    Path("/kaggle/input/datasets/mashrurfaiyaz28/ciot2023-2"),
]
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "CICIoT2023 dataset not found. In Kaggle, Add Input -> "
        "mashrurfaiyaz28/ciot2023-2, then rerun this notebook."
    )

OUTPUT_DIR = Path("/kaggle/working/fl_prepared_iid_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT_NAME)
print("Dataset fraction:", dataset_frac)
print("Number of IID clients:", NUM_CLIENTS)
print("Raw data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Cell 04: Reproducibility and RAM helper
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)


def show_ram(prefix="RAM"):
    process_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    system = psutil.virtual_memory()
    print(f"{prefix} | Process: {process_gb:.2f} GB | System used: {system.percent:.1f}%")


set_seed(SEED)
show_ram("Startup")


In [ ]:
# Cell 05: Discover CSV shards and reproduce the centralized label/feature definition
all_files = sorted(DATA_DIR.rglob("Merged*.csv"))
if not all_files:
    all_files = sorted(DATA_DIR.rglob("*.csv"))
if not all_files:
    raise FileNotFoundError(f"No CSV files found under {DATA_DIR}")

header = pd.read_csv(all_files[0], nrows=0)
if "Label" not in header.columns:
    raise ValueError("Expected a 'Label' column in CICIoT2023 CSV files.")

feature_cols = [c for c in header.columns if c != "Label"]
dtype_map = {c: np.float32 for c in feature_cols}

# Exact stable class order used by the verified centralized full-data run.
classes = [
    'BACKDOOR_MALWARE', 'BENIGN', 'BROWSERHIJACKING', 'COMMANDINJECTION',
    'DDOS-ACK_FRAGMENTATION', 'DDOS-HTTP_FLOOD', 'DDOS-ICMP_FLOOD',
    'DDOS-ICMP_FRAGMENTATION', 'DDOS-PSHACK_FLOOD', 'DDOS-RSTFINFLOOD',
    'DDOS-SLOWLORIS', 'DDOS-SYNONYMOUSIP_FLOOD', 'DDOS-SYN_FLOOD',
    'DDOS-TCP_FLOOD', 'DDOS-UDP_FLOOD', 'DDOS-UDP_FRAGMENTATION',
    'DICTIONARYBRUTEFORCE', 'DNS_SPOOFING', 'DOS-HTTP_FLOOD',
    'DOS-SYN_FLOOD', 'DOS-TCP_FLOOD', 'DOS-UDP_FLOOD',
    'MIRAI-GREETH_FLOOD', 'MIRAI-GREIP_FLOOD', 'MIRAI-UDPPLAIN',
    'MITM-ARPSPOOFING', 'RECON-HOSTDISCOVERY', 'RECON-OSSCAN',
    'RECON-PINGSWEEP', 'RECON-PORTSCAN', 'SQLINJECTION',
    'UPLOADING_ATTACK', 'VULNERABILITYSCAN', 'XSS'
]
label_map = {label: i for i, label in enumerate(classes)}

if len(feature_cols) != 39:
    raise ValueError(f"Expected 39 features, found {len(feature_cols)}")
if len(classes) != 34:
    raise ValueError(f"Expected 34 classes, found {len(classes)}")

# Same deterministic file-fraction rule as the centralized baseline.
rng = np.random.default_rng(SEED)
if dataset_frac >= 1.0:
    files = all_files
else:
    n_files = max(1, int(round(len(all_files) * dataset_frac)))
    chosen_idx = np.sort(rng.choice(len(all_files), size=n_files, replace=False))
    files = [all_files[i] for i in chosen_idx]

effective_file_frac = len(files) / len(all_files)

print("Total CSV files:", len(all_files))
print("Selected CSV files:", len(files))
print("Effective file fraction:", f"{effective_file_frac:.3f}")
print("Features:", len(feature_cols))
print("Classes:", len(classes))


In [ ]:
# Cell 06: Load and clean selected CSV shards exactly like the centralized baseline
load_start = time.perf_counter()
X_parts, y_parts = [], []
rows_raw_total = 0
rows_clean_total = 0

for i, file in enumerate(files, start=1):
    part = pd.read_csv(file, dtype=dtype_map)
    raw_rows = len(part)
    rows_raw_total += raw_rows

    part.replace([np.inf, -np.inf], np.nan, inplace=True)
    part.dropna(inplace=True)
    clean_rows = len(part)
    rows_clean_total += clean_rows

    unknown = set(part["Label"].dropna().unique()) - set(label_map)
    if unknown:
        raise ValueError(f"Unknown labels found in {file.name}: {sorted(unknown)}")

    y_part = part["Label"].map(label_map).to_numpy(dtype=np.int64, copy=True)
    X_part = part[feature_cols].to_numpy(dtype=np.float32, copy=True)

    X_parts.append(X_part)
    y_parts.append(y_part)

    print(
        f"[{i:02d}/{len(files):02d}] {file.name} | "
        f"raw={raw_rows:,} | kept={clean_rows:,}"
    )

    del part, X_part, y_part
    gc.collect()

print("\nCombining rows...")
X = np.concatenate(X_parts, axis=0)
y = np.concatenate(y_parts, axis=0)
del X_parts, y_parts
gc.collect()

load_seconds = time.perf_counter() - load_start

print("Raw rows read:", f"{rows_raw_total:,}")
print("Clean rows:", f"{len(y):,}")
print("X shape:", X.shape, "| dtype:", X.dtype)
print("y shape:", y.shape, "| dtype:", y.dtype)
print("Load time:", f"{load_seconds:.2f} s")
show_ram("After loading")


In [ ]:
# Cell 07: Validate dataset and class coverage
if not np.isfinite(X).all():
    raise ValueError("Feature matrix contains non-finite values after cleaning.")

counts = np.bincount(y, minlength=len(classes))
missing = [classes[i] for i, count in enumerate(counts) if count == 0]
if missing:
    raise ValueError(
        "The selected data does not contain all 34 classes. "
        f"Missing: {missing}. Increase dataset_frac."
    )

class_distribution_df = pd.DataFrame({
    "class_id": np.arange(len(classes)),
    "class_name": classes,
    "count": counts,
    "percent": 100.0 * counts / len(y),
})

display(class_distribution_df)
print("Smallest class:", f"{counts.min():,}")
print("Largest class:", f"{counts.max():,}")


In [ ]:
# Cell 08: Reproduce the centralized 80/20 stratified train-test split
indices = np.arange(len(y), dtype=np.int32)
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)
del indices

# Keep positions inside the training split. Client files will store these positions,
# so the FedAvg notebook can index a materialized X_train directly and efficiently.
y_train_labels = np.ascontiguousarray(y[train_idx], dtype=np.int64)

print("Training samples:", f"{len(train_idx):,}")
print("Testing samples: ", f"{len(test_idx):,}")
print("Train fraction:", f"{len(train_idx) / len(y):.4f}")
print("Test fraction: ", f"{len(test_idx) / len(y):.4f}")
show_ram("After train/test split")


In [ ]:
# Cell 09: Create 10 IID clients from TRAINING data only
# Each class is independently shuffled and split across all clients.
# This gives every client nearly the same class distribution.
client_parts = [[] for _ in range(NUM_CLIENTS)]
rng = np.random.default_rng(SEED)

for class_id in range(len(classes)):
    class_positions = np.flatnonzero(y_train_labels == class_id).astype(np.int32, copy=False)
    rng.shuffle(class_positions)

    for client_id, split in enumerate(np.array_split(class_positions, NUM_CLIENTS)):
        client_parts[client_id].append(split.astype(np.int32, copy=False))

client_positions = []
for client_id in range(NUM_CLIENTS):
    positions = np.concatenate(client_parts[client_id]).astype(np.int32, copy=False)
    rng.shuffle(positions)
    client_positions.append(positions)

del client_parts
gc.collect()

# Structural verification: every training row must belong to exactly one client.
owner_count = np.zeros(len(train_idx), dtype=np.uint8)
for positions in client_positions:
    owner_count[positions] += 1

if not np.all(owner_count == 1):
    bad = int(np.count_nonzero(owner_count != 1))
    raise RuntimeError(f"IID partition verification failed for {bad:,} training rows.")
del owner_count

client_summary = []
for client_id, positions in enumerate(client_positions):
    client_counts = np.bincount(y_train_labels[positions], minlength=len(classes))
    client_summary.append({
        "client_id": client_id,
        "samples": int(len(positions)),
        "min_class_count": int(client_counts.min()),
        "max_class_count": int(client_counts.max()),
    })

client_summary_df = pd.DataFrame(client_summary)
display(client_summary_df)
print("Total rows across clients:", f"{sum(len(x) for x in client_positions):,}")


In [ ]:
# Cell 10: Verify IID class percentages across clients
# The table shows percentages, making it easy to confirm that client distributions match.
rows = []
for client_id, positions in enumerate(client_positions):
    counts_client = np.bincount(y_train_labels[positions], minlength=len(classes))
    percentages = 100.0 * counts_client / len(positions)
    for class_id, pct in enumerate(percentages):
        rows.append({
            "client_id": client_id,
            "class_id": class_id,
            "class_name": classes[class_id],
            "count": int(counts_client[class_id]),
            "percent": float(pct),
        })

client_class_distribution_df = pd.DataFrame(rows)

# Maximum percentage-point spread of the same class across clients.
spread_df = (
    client_class_distribution_df
    .groupby(["class_id", "class_name"])["percent"]
    .agg(["min", "max"])
    .reset_index()
)
spread_df["spread_pp"] = spread_df["max"] - spread_df["min"]

display(spread_df.sort_values("spread_pp", ascending=False).head(10))
print("Largest class-distribution spread across clients:", f"{spread_df['spread_pp'].max():.6f} percentage points")


In [ ]:
# Cell 11: Fit StandardScaler on TRAINING rows only, in chunks
# Chunked partial_fit avoids materializing a second multi-GB X_train copy in this preparation notebook.
scale_start = time.perf_counter()
scaler = StandardScaler()

for chunk_no, start in enumerate(range(0, len(train_idx), SCALER_CHUNK_SIZE), start=1):
    end = min(start + SCALER_CHUNK_SIZE, len(train_idx))
    chunk = np.asarray(X[train_idx[start:end]], dtype=np.float32)
    scaler.partial_fit(chunk)
    del chunk

    if chunk_no % 20 == 0 or end == len(train_idx):
        print(f"Scaler fitted on {end:,}/{len(train_idx):,} training rows")

scaler_seconds = time.perf_counter() - scale_start
scaler_mean = scaler.mean_.astype(np.float64, copy=True)
scaler_scale = scaler.scale_.astype(np.float64, copy=True)

if not np.isfinite(scaler_mean).all() or not np.isfinite(scaler_scale).all():
    raise ValueError("Non-finite StandardScaler statistics found.")
if np.any(scaler_scale == 0):
    raise ValueError("At least one feature has zero StandardScaler scale.")

print("Scaler mean shape:", scaler_mean.shape)
print("Scaler scale shape:", scaler_scale.shape)
print("Scaler fit time:", f"{scaler_seconds:.2f} s")


In [ ]:
# Cell 12: Save prepared FL artifact
save_start = time.perf_counter()

# Large arrays
np.save(OUTPUT_DIR / "X.npy", X, allow_pickle=False)
np.save(OUTPUT_DIR / "y.npy", y, allow_pickle=False)

# Fixed split
np.save(OUTPUT_DIR / "train_idx.npy", train_idx, allow_pickle=False)
np.save(OUTPUT_DIR / "test_idx.npy", test_idx, allow_pickle=False)

# Client positions are relative to train_idx / materialized X_train.
for client_id, positions in enumerate(client_positions):
    np.save(OUTPUT_DIR / f"client_{client_id:02d}_positions.npy", positions, allow_pickle=False)

# Preprocessing metadata
np.save(OUTPUT_DIR / "scaler_mean.npy", scaler_mean, allow_pickle=False)
np.save(OUTPUT_DIR / "scaler_scale.npy", scaler_scale, allow_pickle=False)

with open(OUTPUT_DIR / "classes.json", "w", encoding="utf-8") as f:
    json.dump(classes, f, indent=2)

with open(OUTPUT_DIR / "feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

class_distribution_df.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)
client_summary_df.to_csv(OUTPUT_DIR / "client_summary.csv", index=False)
client_class_distribution_df.to_csv(OUTPUT_DIR / "client_class_distribution.csv", index=False)

metadata = {
    "artifact_type": ARTIFACT_TYPE,
    "experiment_name": EXPERIMENT_NAME,
    "source_dataset": "mashrurfaiyaz28/ciot2023-2",
    "dataset_frac_requested": float(dataset_frac),
    "dataset_file_fraction_effective": float(effective_file_frac),
    "seed": SEED,
    "test_size": TEST_SIZE,
    "num_clients": NUM_CLIENTS,
    "partition": "IID stratified by class across training split",
    "raw_rows_read": int(rows_raw_total),
    "clean_rows": int(len(y)),
    "train_samples": int(len(train_idx)),
    "test_samples": int(len(test_idx)),
    "num_features": int(len(feature_cols)),
    "num_classes": int(len(classes)),
    "client_sizes": [int(len(x)) for x in client_positions],
    "scaler": {
        "type": "StandardScaler",
        "fit_scope": "training split only",
        "fit_method": "chunked partial_fit",
        "chunk_size": SCALER_CHUNK_SIZE,
    },
    "centralized_alignment": {
        "architecture": "39-128-64-34",
        "seed": 42,
        "test_size": 0.20,
        "learning_rate": 0.001,
        "batch_size": 8192,
        "centralized_epochs": 3,
        "note": "With dataset_frac=1.0 this uses the same cleaned full-data scope and split rule as the completed centralized baseline.",
    },
    "selected_csv_files": [p.name for p in files],
    "runtime_seconds": {
        "loading": float(load_seconds),
        "scaler_fit": float(scaler_seconds),
    },
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

save_seconds = time.perf_counter() - save_start
print("Saved prepared FL artifact in", f"{save_seconds:.2f} s")


In [ ]:
# Cell 13: Final verification and file sizes
required_files = [
    "X.npy", "y.npy", "train_idx.npy", "test_idx.npy",
    "scaler_mean.npy", "scaler_scale.npy", "classes.json",
    "feature_cols.json", "metadata.json",
] + [f"client_{i:02d}_positions.npy" for i in range(NUM_CLIENTS)]

missing_files = [name for name in required_files if not (OUTPUT_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Prepared artifact is incomplete. Missing: {missing_files}")

print("\nPrepared artifact files:")
total_bytes = 0
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        size = path.stat().st_size
        total_bytes += size
        print(f"- {path.name:<38} {size / 1024**2:10.2f} MB")

print(f"\nTotal output size: {total_bytes / 1024**3:.2f} GB")
print("Artifact directory:", OUTPUT_DIR)
print("\nNEXT STEP:")
print("1. In Kaggle, Save Version for this notebook with outputs enabled.")
print("2. Open fedavg-iid-optimized-v2.ipynb.")
print("3. Add THIS notebook's saved output as an input.")
print("4. Enable a GPU (Tesla T4 or better) and run all cells.")


## What not to change for the main comparison

For the main centralized-vs-FedAvg comparison, keep `dataset_frac = 1.00`, `SEED = 42`, `TEST_SIZE = 0.20`, and `NUM_CLIENTS = 10` unchanged. This keeps the FL data scope and test split aligned with the completed full centralized baseline.

A smaller `dataset_frac` is useful only for debugging the pipeline; its metrics should not be compared directly with the full-data centralized result.
